# LCP Anomaly Root-Cause Report Pipeline (v6.1)

Fixes the failures observed running v6 against `qwen2.5:7b-instruct`, and adds a
client path for **Qwen3-14B-AWQ served via vLLM** (currently disabled).

| Symptom in the v6 run | Cause | v6.1 fix |
|---|---|---|
| `rejected` with every violation list empty | The only blocking condition left was `"## Executive Summary" in cand` — a case-sensitive literal check the model missed by writing a heading variant. It was never logged. | `normalize_draft()` absorbs formatting variance (code fences, `<think>` blocks, `**bold**`, `###`, numbered headings); structure check is now a case-insensitive regex; **the condition is logged** |
| Report said `("unpacked")` with **no example URL** | `render_page_tokens` only substitutes `⟦PG:…⟧` tokens — the 7B model never emitted them, so nothing happened | `inject_segment_urls()` no longer depends on the model: tokens → plain-name matching → guaranteed "Page Sections Referenced" block, then `verify_urls_present()` asserts none are missing |
| "P75 increased by **56.1ms**" (actual p75 change: 174ms); "**79.8% vs 79.8%**" | Both numbers exist in findings, so the whitelist passed them. A whitelist proves a number *exists*, not that it is used *correctly*. | `build_narrative_facts()` pre-writes every load-bearing sentence in Python; `check_number_binding()` verifies numbers match their metric; `check_degenerate_comparison()` catches "X vs X" |
| `2023` treated as auto-fixable | Section-based soft/hard split let a hallucinated year through as soft | A 4-digit year is a **hard** violation in any section |

Serving note: Qwen3/vLLM settings are kept for later but currently disabled;
the active default follows v6-compatible Ollama settings.
(If re-enabled later, `normalize_draft` still strips any `<think>` blocks.)

In [ ]:
# Cell 1 — Configuration (no secrets in this notebook)
import os
from pathlib import Path

CSV_FILENAME   = os.getenv("ANOMALY_CSV", "sample_data_7-20_2357_largestcontentfulpaint_iter5_win1_inter5.csv")
PROCESSED_DIR  = Path(os.getenv("PROCESSED_DIR", "/opt/perf-analytics/processed"))
METADATA_PATH  = PROCESSED_DIR / (Path(CSV_FILENAME).stem + ".meta.json")   # sidecar from upstream

TIMER_COL, LABEL_COL = "timer", "label"
PRIMARY_DIM    = "page_group"                       # derived below; the first drilldown axis
SECONDARY_DIMS = ["country", "connectiontype", "deviceType", "isp"]
CATEGORICAL    = ["page_group","country","deviceType","os","browser","protocol",
                  "connectiontype","origin_flag","isp","landingpage","paidmedia","referrer_present"]
DELIVERY_NUMERIC = ["deviceMemory","rtt","cacherate","cdncacherate","transferbyte",
                    "bodysize","requestcount","origintime","edgetime"]

ARTIFACT_MS      = 60_000     # beacons above this are background-tab artifacts
SEVERITY_FLOOR_MS= 50         # p75 deltas below this never alert
SAMPLE_DRIFT_TOL = 3.0        # % allowed drift between sample and source stats
MIN_SEG_N        = 300        # min beacons per window for a segment to be reported
MIN_SHARE_PP     = 1.0        # min share change to qualify as a mover
MAX_FOCUS        = 3          # max problem sections listed in detail (rest summarized)        # min share change to qualify as a mover

# [TEMP DISABLED] Qwen3-14B-AWQ via vLLM (kept for later re-enable)
# LLM_BACKEND  = os.getenv("LLM_BACKEND", "openai")            # "openai" (vLLM) | "ollama"
# LLM_URL      = os.getenv("LLM_URL", "http://localhost:8000/v1/chat/completions")
# LLM_MODEL    = os.getenv("LLM_MODEL", "Qwen/Qwen3-14B-AWQ")
# LLM_API_KEY  = os.getenv("LLM_API_KEY", "EMPTY")             # vLLM ignores the value
# # Qwen3 non-thinking sampling recommendation
# LLM_TEMPERATURE = float(os.getenv("LLM_TEMPERATURE", "0.7"))
# LLM_TOP_P       = float(os.getenv("LLM_TOP_P", "0.8"))
# LLM_ENABLE_THINKING = os.getenv("LLM_ENABLE_THINKING", "0") == "1"
# Active LLM defaults: v6-compatible Ollama setup
LLM_BACKEND  = os.getenv("LLM_BACKEND", "ollama")
LLM_URL      = os.getenv("LLM_URL", os.getenv("OLLAMA_URL", "http://localhost:11434/api/generate"))
LLM_MODEL    = os.getenv("LLM_MODEL", os.getenv("OLLAMA_MODEL", "qwen2.5:7b-instruct"))
LLM_API_KEY  = os.getenv("LLM_API_KEY", "")             # not used by Ollama
LLM_TEMPERATURE = float(os.getenv("LLM_TEMPERATURE", "0.7"))
LLM_TOP_P       = float(os.getenv("LLM_TOP_P", "0.8"))
LLM_ENABLE_THINKING = os.getenv("LLM_ENABLE_THINKING", "0") == "1"
DRY_RUN_EMAIL= os.getenv("DRY_RUN_EMAIL", "1") == "1"   # default: never send by accident

# Human-label overrides for customer wording (optional, generic pipeline works without)
FEATURE_LABEL_OVERRIDES = {}
print(f"config ok — csv={CSV_FILENAME}, dry_run_email={DRY_RUN_EMAIL}")

In [ ]:
# Cell 2 — Core statistical modules (v6.1)
"""Core deterministic analysis functions for the LCP anomaly report pipeline.

Design principles (generic, not dataset-specific):
- No hardcoded page names, countries, or segments. All "top movers" are discovered.
- Every reported number is computed here and carried into a findings dict; the
  LLM only verbalizes findings and is validated against the allowed-number set.
"""
import re
import numpy as np
import pandas as pd
from scipy import stats

RNG = np.random.default_rng(42)

# ---------------------------------------------------------------- page groups
def derive_page_group(url_series: pd.Series, top_k: int = 15) -> pd.Series:
    """Extract a coarse page group from URL paths, generic to any site.

    Takes the first path segment after an (optional) short locale segment.
    Groups outside the top_k by traffic volume are folded into 'other'.
    """
    def _seg(u):
        if not isinstance(u, str):
            return "unknown"
        m = re.match(r"^https?://[^/]+/(.*)$", u)
        if not m:
            return "unknown"
        parts = [p for p in m.group(1).split("/") if p]
        if not parts:
            return "home"
        # treat a leading 1-5 char alnum segment as locale (uk, in, sec, latin...)
        if len(parts) >= 2 and re.fullmatch(r"[a-z_\-]{1,5}", parts[0]):
            return parts[1].split("?")[0] or "home"
        return parts[0].split("?")[0] or "home"
    seg = url_series.map(_seg)
    top = seg.value_counts().head(top_k).index
    return seg.where(seg.isin(top), "other")


# ------------------------------------------------------------- window stats
def compute_window_stats(df, timer_col="timer", label_col="label",
                         n_boot=300):
    out = {}
    for lab, key in [(0, "normal"), (1, "anomaly")]:
        s = df.loc[df[label_col] == lab, timer_col]
        out[key] = {
            "count": int(len(s)),
            "mean": round(float(s.mean()), 1),
            "p50": round(float(s.quantile(.50)), 1),
            "p75": round(float(s.quantile(.75)), 2),
            "p90": round(float(s.quantile(.90)), 1),
            "p95": round(float(s.quantile(.95)), 1),
            "p99": round(float(s.quantile(.99)), 1),
        }
    d75 = out["anomaly"]["p75"] - out["normal"]["p75"]
    out["delta_p75_ms"] = round(d75, 2)
    out["delta_p75_pct"] = round(d75 / out["normal"]["p75"] * 100, 2)

    a = df.loc[df[label_col] == 1, timer_col].to_numpy()
    n = df.loc[df[label_col] == 0, timer_col].to_numpy()
    mw = stats.mannwhitneyu(a, n, alternative="two-sided")
    out["mannwhitney_p"] = float(mw.pvalue)

    # bootstrap CI on the p75 difference
    boots = np.empty(n_boot)
    for i in range(n_boot):
        boots[i] = (np.quantile(RNG.choice(a, len(a)), .75)
                    - np.quantile(RNG.choice(n, len(n)), .75))
    lo, hi = np.percentile(boots, [2.5, 97.5])
    out["delta_p75_ci95"] = [round(float(lo), 1), round(float(hi), 1)]
    out["delta_significant"] = bool(lo > 0 or hi < 0)
    return out


def classify_severity(win, abs_floor_ms=50):
    """Severity from the p75 delta, gated by significance and an absolute floor."""
    d, pct = win["delta_p75_ms"], win["delta_p75_pct"]
    if not win["delta_significant"] or abs(d) < abs_floor_ms:
        return "none", "No statistically meaningful change in p75."
    if pct < 0:
        return "improved", "p75 improved versus the normal window."
    if pct < 2:
        return "info", "Very small p75 increase; monitoring only."
    if pct < 5:
        return "low", "Small but significant p75 increase."
    if pct < 15:
        return "warning", "Meaningful p75 degradation."
    return "critical", "Severe p75 degradation."


# ---------------------------------------------------------------- outliers
def audit_outliers(df, timer_col="timer", label_col="label",
                   artifact_ms=60_000):
    """Quantify extreme-tail beacons (likely background-tab artifacts) and
    their influence on the mean, per window."""
    res = {"artifact_threshold_ms": artifact_ms, "windows": {}}
    for lab, key in [(0, "normal"), (1, "anomaly")]:
        s = df.loc[df[label_col] == lab, timer_col]
        art = s[s > artifact_ms]
        mean_all = float(s.mean())
        mean_clean = float(s[s <= artifact_ms].mean()) if (s <= artifact_ms).any() else np.nan
        res["windows"][key] = {
            "artifact_count": int(len(art)),
            "artifact_share_pct": round(len(art) / len(s) * 100, 3),
            "max_timer_ms": int(s.max()),
            "mean_all_ms": round(mean_all, 1),
            "mean_excl_artifacts_ms": round(mean_clean, 1),
            "mean_inflation_ms": round(mean_all - mean_clean, 1),
        }
    return res


# ------------------------------------------------------------ decomposition
def mix_within_decomposition(df, dim, timer_col="timer", label_col="label",
                             stat="mean"):
    """Oaxaca-style decomposition of the overall change in `stat` of timer
    into composition (mix) and within-segment effects along `dim`.
    `dim` may be a column name or a list of columns (joint segments)."""
    if isinstance(dim, (list, tuple)):
        key = df[list(dim)].astype(str).agg("|".join, axis=1)
        dim_name = "x".join(dim)
    else:
        key, dim_name = df[dim].astype(str), dim
    agg = "mean" if stat == "mean" else "median"
    g0 = df[df[label_col] == 0].groupby(key[df[label_col] == 0])[timer_col].agg(["count", agg])
    g1 = df[df[label_col] == 1].groupby(key[df[label_col] == 1])[timer_col].agg(["count", agg])
    j = g0.join(g1, lsuffix="0", rsuffix="1", how="outer").fillna(0)
    j["s0"] = j["count0"] / max(j["count0"].sum(), 1)
    j["s1"] = j["count1"] / max(j["count1"].sum(), 1)
    mix = float(((j.s1 - j.s0) * j[f"{agg}0"]).sum())
    within = float((j.s1 * (j[f"{agg}1"] - j[f"{agg}0"])).sum())
    total = float(df.loc[df[label_col] == 1, timer_col].agg(agg)
                  - df.loc[df[label_col] == 0, timer_col].agg(agg))
    return {"dim": dim_name, "stat": stat, "total_delta_ms": round(total, 1),
            "mix_effect_ms": round(mix, 1), "within_effect_ms": round(within, 1)}


def top_movers(df, dim, timer_col="timer", label_col="label",
               min_n=200, top=5):
    """Discover segments whose traffic share or internal p75 moved the most."""
    n0 = (df[label_col] == 0).sum()
    n1 = (df[label_col] == 1).sum()
    rows = []
    for val, sub in df.groupby(dim, dropna=False):
        c0 = (sub[label_col] == 0).sum()
        c1 = (sub[label_col] == 1).sum()
        if c0 < min_n or c1 < min_n:
            continue
        p75_0 = sub.loc[sub[label_col] == 0, timer_col].quantile(.75)
        p75_1 = sub.loc[sub[label_col] == 1, timer_col].quantile(.75)
        med_0 = sub.loc[sub[label_col] == 0, timer_col].median()
        med_1 = sub.loc[sub[label_col] == 1, timer_col].median()
        rows.append({
            "segment": str(val), "n_normal": int(c0), "n_anomaly": int(c1),
            "share_normal_pct": round(c0 / n0 * 100, 2),
            "share_anomaly_pct": round(c1 / n1 * 100, 2),
            "share_delta_pp": round(c1 / n1 * 100 - c0 / n0 * 100, 2),
            "p75_normal": round(float(p75_0), 1),
            "p75_anomaly": round(float(p75_1), 1),
            "p75_delta_ms": round(float(p75_1 - p75_0), 1),
            "median_normal": round(float(med_0), 1),
            "median_anomaly": round(float(med_1), 1),
        })
    t = pd.DataFrame(rows)
    if t.empty:
        return {"dim": dim, "share_movers": [], "perf_movers": []}
    overall_p75_normal = df.loc[df[label_col] == 0, timer_col].quantile(.75)
    t["slow_segment"] = t["p75_normal"] > overall_p75_normal
    share_movers = t.reindex(t["share_delta_pp"].abs()
                             .sort_values(ascending=False).index).head(top)
    perf_movers = t.reindex(t["p75_delta_ms"].abs()
                            .sort_values(ascending=False).index).head(top)
    return {"dim": dim,
            "share_movers": share_movers.to_dict("records"),
            "perf_movers": perf_movers.to_dict("records")}


def localization_check(df, dim, segment, timer_col="timer", label_col="label"):
    """Is the sitewide p75 change fully explained by one segment?
    Reports overall / segment-only / segment-excluded p75 shifts."""
    def p75s(d):
        return (round(float(d.loc[d[label_col] == 0, timer_col].quantile(.75)), 1),
                round(float(d.loc[d[label_col] == 1, timer_col].quantile(.75)), 1))
    inseg = df[df[dim].astype(str) == str(segment)]
    exseg = df[df[dim].astype(str) != str(segment)]
    o0, o1 = p75s(df); i0, i1 = p75s(inseg); e0, e1 = p75s(exseg)
    return {"dim": dim, "segment": str(segment),
            "overall_p75": [o0, o1], "segment_p75": [i0, i1],
            "excluded_p75": [e0, e1],
            # localized: the rest of the site did not move in the same
            # direction by more than 35% of the overall shift (signed test,
            # so an improvement outside the segment still counts as localized)
            "localized": bool((e1 - e0) < 0.35 * (o1 - o0) if (o1 - o0) > 0
                              else (e1 - e0) > 0.35 * (o1 - o0) if (o1 - o0) < 0
                              else False)}


# ------------------------------------------------------- behavioral signals
def behavior_signals(df, label_col="label",
                     landing_col="landingpage", referrer_col="referrer",
                     clientcache_col="cacherate", transfer_col="transferbyte",
                     conn_col="connectiontype", cellular_value="Cellular"):
    """Generic audience-composition signals. Flags a 'new-visitor influx'
    pattern when session-entry share rises while referrer presence and
    client cache rate fall together. Column names are parameters so the
    module ports to other beacon schemas."""
    sig = {}
    for lab, key in [(0, "normal"), (1, "anomaly")]:
        d = df[df[label_col] == lab]
        sig[key] = {
            "landing_share_pct": round(float((d[landing_col] == True).mean()) * 100, 1)
                                  if landing_col in d else None,
            "referrer_present_pct": round(float(d[referrer_col].notna().mean()) * 100, 1)
                                  if referrer_col in d else None,
            "client_cacherate_median": round(float(d[clientcache_col].median()), 1)
                                  if clientcache_col in d else None,
            "transferbyte_median": int(d[transfer_col].median())
                                  if transfer_col in d else None,
            "cellular_share_pct": round(float((d[conn_col] == cellular_value).mean()) * 100, 1)
                                  if conn_col in d else None,
        }
    n, a = sig["normal"], sig["anomaly"]
    checks = []
    if n["landing_share_pct"] is not None:
        checks.append(a["landing_share_pct"] - n["landing_share_pct"] > 2)
    if n["referrer_present_pct"] is not None:
        checks.append(n["referrer_present_pct"] - a["referrer_present_pct"] > 2)
    if n["client_cacherate_median"] not in (None, 0):
        checks.append(a["client_cacherate_median"]
                      < 0.8 * n["client_cacherate_median"])
    sig["new_visitor_influx"] = bool(checks and sum(checks) >= 2)
    return sig


# -------------------------------------------------------- delivery health
def delivery_health(df, label_col="label", tol_pct=15,
                    metrics=("edgetime", "origintime", "cdncacherate"),
                    origin_flag_col="origin_flag"):
    """CDN/origin health check: verdict is 'clean' unless a delivery metric
    degrades beyond tolerance in the anomaly window."""
    res = {"metrics": {}, "issues": []}
    for m in metrics:
        if m not in df:
            continue
        m0 = float(df.loc[df[label_col] == 0, m].median())
        m1 = float(df.loc[df[label_col] == 1, m].median())
        res["metrics"][m] = {"normal_median": round(m0, 1),
                             "anomaly_median": round(m1, 1)}
        worse = (m1 > m0 * (1 + tol_pct / 100)) if m != "cdncacherate" \
            else (m1 < m0 * (1 - tol_pct / 100))
        base_floor = 20 if m != "cdncacherate" else 0
        if worse and max(m0, m1) > base_floor:
            res["issues"].append(m)
    if origin_flag_col in df:
        o0 = float((df.loc[df[label_col] == 0, origin_flag_col] == "Y").mean()) * 100
        o1 = float((df.loc[df[label_col] == 1, origin_flag_col] == "Y").mean()) * 100
        res["metrics"]["origin_traffic_share_pct"] = {
            "normal_median": round(o0, 1), "anomaly_median": round(o1, 1)}
        if o1 > o0 + 5:
            res["issues"].append("origin_traffic_share")
    res["verdict"] = "degraded" if res["issues"] else "clean"
    return res


# ============================ v3 additions ============================
import re as _re

def representative_urls(frame, group_col="page_group", url_col="url"):
    """Most canonical concrete URL per page group. Prefers the shallowest
    path depth (fewest segments = landing/section root), tie-broken by
    frequency. Query strings and fragments are stripped. Generic to any site."""
    mapping = {}
    if url_col not in frame:
        return mapping
    for grp, sub in frame.groupby(group_col):
        urls = (sub[url_col].dropna().astype(str)
                .map(lambda u: u.split("?")[0].split("#")[0]))
        if urls.empty:
            continue
        vc = urls.value_counts()
        cand = vc.head(8).index.tolist()
        def _depth(u):
            return len([p for p in _re.sub(r"^https?://[^/]+", "", u).split("/") if p])
        mapping[str(grp)] = sorted(cand, key=lambda u: (_depth(u), -int(vc[u])))[0]
    return mapping


def composite_verdict_flags(focus, within_floor_ms=80):
    """Detect whether the focus segment ALSO degraded on its own (a real
    within-segment regression) on top of gaining traffic share. Measured on
    the focus segment's OWN median and p75 deltas — not the site-wide
    decomposition, which nets across many groups and hides local regressions.
    Upgrades a pure 'traffic_mix_shift' verdict to a composite one so the
    within-segment signal is never lost."""
    if not focus:
        return {"within_regression": False, "focus_median_delta_ms": 0.0,
                "focus_p75_delta_ms": 0.0}
    med_d = float(focus.get("median_anomaly", 0) - focus.get("median_normal", 0))
    p75_d = float(focus.get("p75_delta_ms", 0))
    return {"within_regression": bool(med_d >= within_floor_ms
                                      or p75_d >= within_floor_ms),
            "focus_median_delta_ms": round(med_d, 1),
            "focus_p75_delta_ms": round(p75_d, 1)}


# ============================ v4 additions ============================
def select_focus_segments(primary_movers, overall_win, dim,
                          min_share_pp=1.0, min_p75_delta=80,
                          min_anom_share=1.5, max_focus=3):
    """v4: return a LIST of problem segments, not one. A segment qualifies if
    EITHER (a) it is slower-than-site AND gained meaningful traffic share, OR
    (b) its own p75 degraded materially while carrying non-trivial traffic.
    Ranked by contribution to the sitewide p75 rise; capped at max_focus with
    the remainder summarized. Fully generic — no segment names referenced."""
    share = {r["segment"]: r for r in primary_movers["share_movers"]}
    perf = {r["segment"]: r for r in primary_movers["perf_movers"]}
    universe = {**perf, **share}     # union of both mover views

    picked = {}
    for seg, r in universe.items():
        gained_share = (r["share_delta_pp"] >= min_share_pp and r.get("slow_segment"))
        self_regressed = (r["p75_delta_ms"] >= min_p75_delta
                          and r["share_anomaly_pct"] >= min_anom_share)
        if gained_share or self_regressed:
            # contribution proxy to sitewide p75 rise: share-growth pull + own worsening
            contrib = (max(r["share_delta_pp"], 0) / 100.0) * r["p75_normal"] \
                      + (r["share_anomaly_pct"] / 100.0) * max(r["p75_delta_ms"], 0)
            picked[seg] = {**r,
                           "gained_share": bool(gained_share),
                           "self_regressed": bool(self_regressed),
                           "contribution_score": round(float(contrib), 1)}
    ranked = sorted(picked.values(), key=lambda x: -x["contribution_score"])
    return {"focus_list": ranked[:max_focus],
            "additional_count": max(0, len(ranked) - max_focus),
            "additional_segments": [r["segment"] for r in ranked[max_focus:]],
            "total_qualified": len(ranked)}


def coverage_check(df, dim, focus_segments, timer_col="timer", label_col="label"):
    """What fraction of the sitewide p75 rise is attributable to the chosen
    focus segments? Removes those segments and re-measures the residual p75
    shift. Low coverage => the story is incomplete (widen top_k / try another
    dimension)."""
    def p75(d, lab):
        return float(d.loc[d[label_col] == lab, timer_col].quantile(.75))
    o0, o1 = p75(df, 0), p75(df, 1)
    overall = o1 - o0
    rest = df[~df[dim].astype(str).isin([str(s) for s in focus_segments])]
    r0, r1 = p75(rest, 0), p75(rest, 1)
    residual = r1 - r0
    explained = overall - residual
    cov = (explained / overall) if abs(overall) > 1e-9 else 0.0
    return {"overall_p75_delta": round(overall, 1),
            "residual_p75_delta": round(residual, 1),
            "explained_p75_delta": round(explained, 1),
            "coverage_ratio": round(float(cov), 2),
            "sufficient": bool(cov >= 0.7)}


def other_bucket_watch(primary_movers, other_label="other", p75_floor=100):
    """Flag when the catch-all 'other' bucket (groups beyond top_k) itself
    shows a material p75 rise — a hidden segment may be the real culprit."""
    for r in primary_movers["perf_movers"] + primary_movers["share_movers"]:
        if r["segment"] == other_label:
            if r["p75_delta_ms"] >= p75_floor:
                return {"flagged": True, "p75_delta_ms": r["p75_delta_ms"],
                        "share_anomaly_pct": r["share_anomaly_pct"],
                        "note": ("The 'other' bucket (page groups beyond the top "
                                 "tracked set) degraded materially; increase top_k "
                                 "to expose the hidden group.")}
            return {"flagged": False}
    return {"flagged": False}


In [ ]:
# Cell 3 — Model modules (evidence generators)
"""Model layer for the anomaly report pipeline.

Two models with distinct, honest roles:
1. Window classifier (context features only, NO timer, NO delivery metrics):
   answers "did the traffic composition change?" — gated by holdout AUC.
   Its SHAP output is labeled as a *composition fingerprint*, never as a
   performance cause.
2. Timer regressor on log1p(timer) (context + delivery metrics):
   answers "what drives LCP?" — the per-feature difference in mean SHAP
   between windows attributes the predicted LCP shift to features.
"""
import numpy as np
import pandas as pd
import xgboost as xgb
import shap
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score

MAX_CAT_CARDINALITY = 20      # top-N categories kept per column, rest folded
SHAP_SAMPLE = 15_000          # rows sampled per window for SHAP


def build_features(df, categorical_cols, numeric_cols):
    """One-hot with cardinality capping. drop_first=False so every category
    is interpretable on its own (no hidden baseline)."""
    X = pd.DataFrame(index=df.index)
    for c in numeric_cols:
        if c in df:
            X[c] = pd.to_numeric(df[c], errors="coerce")
    cats = pd.DataFrame(index=df.index)
    for c in categorical_cols:
        if c not in df:
            continue
        s = df[c].astype(str).fillna("unknown")
        top = s.value_counts().head(MAX_CAT_CARDINALITY).index
        cats[c] = s.where(s.isin(top), "other")
    if len(cats.columns):
        X = pd.concat([X, pd.get_dummies(cats, drop_first=False)], axis=1)
    return X


def window_classifier_fingerprint(df, categorical_cols, label_col="label",
                                  auc_gate=0.60, seed=42):
    """Train label(window) classifier on context features only.
    Returns AUC and, if the gate passes, the top composition-shift features."""
    X = build_features(df, categorical_cols, numeric_cols=[])
    y = df[label_col].astype(int)
    Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.25,
                                          stratify=y, random_state=seed)
    clf = xgb.XGBClassifier(n_estimators=200, max_depth=5, learning_rate=0.1,
                            eval_metric="logloss", n_jobs=4,
                            random_state=seed)
    clf.fit(Xtr, ytr)
    auc = float(roc_auc_score(yte, clf.predict_proba(Xte)[:, 1]))
    result = {"holdout_auc": round(auc, 3), "gate_passed": auc >= auc_gate,
              "fingerprint": []}
    if auc < auc_gate:
        result["note"] = ("Traffic composition of the two windows is nearly "
                          "indistinguishable; composition fingerprint skipped.")
        return result
    idx = np.random.default_rng(seed).choice(
        len(Xte), min(SHAP_SAMPLE, len(Xte)), replace=False)
    sv = shap.TreeExplainer(clf).shap_values(Xte.iloc[idx])
    imp = pd.Series(np.abs(sv).mean(0), index=X.columns)
    for feat, val in imp.sort_values(ascending=False).head(8).items():
        on_share_normal = float(X.loc[y == 0, feat].mean()) * 100
        on_share_anom = float(X.loc[y == 1, feat].mean()) * 100
        result["fingerprint"].append({
            "feature": feat, "mean_abs_shap": round(float(val), 4),
            "share_normal_pct": round(on_share_normal, 2),
            "share_anomaly_pct": round(on_share_anom, 2),
            "share_delta_pp": round(on_share_anom - on_share_normal, 2)})
    return result


def timer_regressor_drivers(df, categorical_cols, numeric_cols,
                            timer_col="timer", label_col="label",
                            artifact_ms=60_000, seed=42, top=10):
    """Regress log1p(timer) on context + delivery features; attribute the
    window-to-window predicted shift via the per-feature mean-SHAP delta.
    Artifact beacons (timer > artifact_ms) are excluded from training so the
    model explains typical user experience rather than background tabs."""
    d = df[df[timer_col] <= artifact_ms].copy()
    X = build_features(d, categorical_cols, numeric_cols)
    y = np.log1p(d[timer_col].astype(float))
    reg = xgb.XGBRegressor(n_estimators=300, max_depth=6, learning_rate=0.08,
                           n_jobs=4, random_state=seed)
    reg.fit(X, y)
    r2 = float(reg.score(X, y))

    rng = np.random.default_rng(seed)
    parts = []
    for lab in (0, 1):
        pos = np.flatnonzero((d[label_col] == lab).to_numpy())
        parts.append(rng.choice(pos, min(SHAP_SAMPLE, len(pos)), replace=False))
    idx = np.concatenate(parts)
    Xs = X.iloc[idx]
    labs = d[label_col].to_numpy()[idx]
    sv = shap.TreeExplainer(reg).shap_values(Xs)

    mean_normal = sv[labs == 0].mean(0)
    mean_anom = sv[labs == 1].mean(0)
    delta = mean_anom - mean_normal          # log-space contribution shift
    order = np.argsort(-np.abs(delta))[:top]
    drivers = []
    for i in order:
        feat = X.columns[i]
        drivers.append({
            "feature": feat,
            "shap_delta_log": round(float(delta[i]), 4),
            "direction": "worsening" if delta[i] > 0 else "improving",
            "value_median_normal": round(float(
                pd.to_numeric(Xs.loc[labs == 0, feat], errors="coerce").median()), 2),
            "value_median_anomaly": round(float(
                pd.to_numeric(Xs.loc[labs == 1, feat], errors="coerce").median()), 2),
        })
    pred_shift_pct = round(float(np.exp(sv[labs == 1].sum(1).mean()
                                        - sv[labs == 0].sum(1).mean()) - 1) * 100, 2)
    return {"train_r2": round(r2, 3),
            "predicted_shift_pct": pred_shift_pct,
            "drivers": drivers}


In [ ]:
# Cell 4 — Findings, verdict, validation layer, email HTML (v6.1)
"""Findings assembly, verdict selection, LLM verbalization and validation."""
import json
import re
import re as _re
import numpy as np


# ------------------------------------------------------------- humanization
def humanize_feature(feat, overrides=None):
    """Generic feature-name -> customer-readable phrase. Works for any
    one-hot 'col_value' name without dataset-specific hardcoding."""
    overrides = overrides or {}
    if feat in overrides:
        return overrides[feat]
    generic = {
        "page_group": "the '{v}' page section",
        "country": "traffic from region '{v}'",
        "isp": "users on network provider '{v}'",
        "connectiontype": "'{v}' network connections",
        "deviceType": "'{v}' devices",
        "os": "'{v}' devices",
        "browser": "'{v}' browser sessions",
        "protocol": "connections over '{v}'",
        "landingpage": "session-entry page views" ,
        "referrer_present": "visits arriving without a referrer",
        "paidmedia": "paid-media traffic",
        "origin_flag": "requests served via origin",
    }
    numeric = {
        "cacherate": "browser cache hit rate",
        "cdncacherate": "CDN cache hit rate",
        "transferbyte": "bytes transferred over the network",
        "bodysize": "total page weight",
        "requestcount": "number of page requests",
        "origintime": "origin response time",
        "edgetime": "edge response time",
        "rtt": "network round-trip time",
        "deviceMemory": "device memory",
    }
    if feat in numeric:
        return numeric[feat]
    for col, tpl in generic.items():
        if feat.startswith(col + "_"):
            return tpl.format(v=feat[len(col) + 1:])
        if feat == col:
            return tpl.format(v="")
    return feat.replace("_", " ")


# --------------------------------------------------------- verdict selection
def select_verdict(severity, decomp_primary, localization, behavior,
                   delivery, outliers, within_flags=None,
                   focus_selection=None, coverage=None):
    """v4: rule-based story selection over a LIST of problem segments.
    Recognizes single vs multiple problem sections and flags incomplete
    coverage so the report never overclaims."""
    art_norm=outliers["windows"]["normal"]["mean_inflation_ms"]
    art_anom=outliers["windows"]["anomaly"]["mean_inflation_ms"]
    outlier_note=(art_norm>100 or art_anom>100)
    within_reg=bool(within_flags and within_flags.get("within_regression"))
    focus_list=(focus_selection or {}).get("focus_list", [])
    n_focus=len(focus_list)
    multi=n_focus>=2
    low_cov=bool(coverage and not coverage.get("sufficient"))

    if severity in ("none","info","improved"):
        s=("No meaningful page-load degradation was found between the two windows.")
        if outlier_note:
            s+=(" A small number of extreme-duration beacons inflate the average; "
                "percentile-based views are recommended.")
        return "no_action", s

    if delivery["verdict"]=="degraded":
        return "delivery_regression", (
            "Delivery-layer metrics degraded in the anomaly window; CDN or origin "
            "behavior should be investigated first.")

    mix=decomp_primary["mix_effect_ms"]; within=decomp_primary["within_effect_ms"]
    mix_dominant=mix>0 and mix>=abs(within)

    # how many focus segments regressed on their own vs merely gained share
    any_self=any(r.get("self_regressed") for r in focus_list)
    any_share=any(r.get("gained_share") for r in focus_list)

    if multi:
        s=(f"The slowdown spans {n_focus} page sections rather than one. ")
        if any_self and any_share:
            s+=("Some grew their share of traffic while others became slower on "
                "their own; the report lists each with its own evidence.")
        elif any_self:
            s+="Several sections became genuinely slower in the anomaly window."
        else:
            s+=("Several slower sections grew their share of traffic while the rest "
                "of the site held steady.")
        code="multi_segment_regression" if any_self else "multi_segment_mix_shift"
    elif mix_dominant and within_reg:
        code="mix_shift_with_local_regression"
        s=("The slowdown has two compounding causes: a slower page section grew its "
           "share of traffic AND that section became genuinely slower on its own in "
           "the anomaly window.")
        if behavior.get("new_visitor_influx"):
            s+=(" The incoming traffic shows a new-visitor pattern (more session "
                "entries, fewer referrers, colder browser caches), consistent with a "
                "campaign or event launch.")
    elif mix_dominant and localization and localization.get("localized"):
        code="traffic_mix_shift"
        s=("The slowdown is a traffic-composition effect: a slower page section grew "
           "its share of traffic while the rest of the site held steady or improved.")
        if behavior.get("new_visitor_influx"):
            s+=(" The incoming traffic shows a new-visitor pattern (more session "
                "entries, fewer referrers, colder browser caches), consistent with a "
                "campaign or event launch.")
    elif mix_dominant:
        code="traffic_mix_shift_broad"
        s=("The slowdown is driven mainly by a shift in traffic composition across "
           "several segments rather than by pages getting slower.")
    else:
        code="segment_regression"
        s=("Specific segments became genuinely slower in the anomaly window; the "
           "change is not explained by traffic composition alone.")

    if low_cov:
        s+=(f" Note: the identified sections explain only "
            f"{int(round(coverage['coverage_ratio']*100))}% of the p75 change; a "
            f"further contributor remains unaccounted for.")
    return code, s


# ------------------------------------------------------- findings container
def collect_numbers(obj, acc=None):
    if acc is None:
        acc = set()
    if isinstance(obj, dict):
        for v in obj.values():
            collect_numbers(v, acc)
    elif isinstance(obj, (list, tuple)):
        for v in obj:
            collect_numbers(v, acc)
    elif isinstance(obj, bool):
        pass
    elif isinstance(obj, (int, float, np.integer, np.floating)):
        acc.add(round(float(obj), 2))
    return acc


def validate_report_numbers(report_text, allowed, small_int_max=15):
    """Every number in the report must exist in the findings (or be a small
    structural integer). Returns the list of unauthorized numbers."""
    bad = []
    for m in re.finditer(r"(?<![\w.])\d[\d,]*(?:\.\d+)?", report_text):
        raw = m.group(0).replace(",", "")
        try:
            v = float(raw)
        except ValueError:
            continue
        if v <= small_int_max and v == int(v):
            continue
        cands = {round(v, 2), round(v, 1), float(int(v))}
        if not any(c in allowed for c in cands):
            # tolerate values that round-trip to an allowed number
            if not any(abs(c - a) <= 0.51 for a in allowed for c in [v]):
                bad.append(m.group(0))
    return sorted(set(bad))


# ------------------------------------------------------------ LLM prompting
def build_llm_prompt(findings_json_text):
    return f"""You are a website performance analyst writing for a business customer.

Your ONLY source of truth is the FINDINGS JSON below. Verbalize it; do not analyze.

Strict rules:
- English only.
- Use ONLY numbers that literally appear in the FINDINGS JSON. Never compute,
  convert, or invent numbers. If unsure, omit the number.
- To refer to any page section, write its token EXACTLY as given in
  findings (e.g. the "token" field, like \u27e6PG:unpacked\u27e7). Do NOT
  rewrite, translate, or expand tokens; copy them verbatim. A later step
  turns each token into a readable name and an example URL.
- CAUSE vs SYMPTOM: items under findings.performance_drivers are candidate
  drivers. Items under findings.associated_symptoms (e.g. browser cache hit
  rate, transferred bytes, session-entry share) are INDICATORS of a traffic
  change, NOT causes. Never call a symptom a "driver", "cause", or "root
  cause". Describe symptoms as evidence of who is visiting, not as reasons
  the site got slower.
- No machine-learning jargon (no SHAP, model, classifier, feature, one-hot).
- Follow the verdict: the report storyline must match findings.verdict.sentence.
- If findings.delivery.verdict is "clean", explicitly reassure the customer
  that CDN and origin infrastructure show no regression.

Output format (Markdown, exactly these sections):
## Executive Summary
(2-3 sentences: the transition sentence in your own words, then the verdict.)
## What Changed
(List EVERY page section in findings.segments.focus_list, each with its token
and its own share/p75 change and role. If findings.coverage.sufficient is
false, state plainly that the listed sections do not fully explain the change.
Present behavior items as audience indicators, never as causes.)
## What Did Not Change
(Delivery-layer health; segments that stayed stable or improved.)
## Recommended Actions
(3-4 practical actions matched to the verdict code.)
## Monitoring Notes
(1-2 sentences on what to watch next.)

FINDINGS JSON:
{findings_json_text}
"""


# ------------------------------------------------------- deterministic fall-back
def render_fallback_report(f):
    """Deterministic report. v4: iterates over ALL focus segments and adds a
    coverage note when the identified sections don't fully explain the change."""
    from_token=f.get("_token_fn", lambda s: f"the '{s}' page section")
    h,v=f["headline"], f["verdict"]
    lines=["## Executive Summary", h["transition_sentence"], v["sentence"], ""]
    lines.append("## What Changed")
    focus_list=f["segments"].get("focus_list") or []
    if focus_list:
        for r in focus_list:
            tok=from_token(r["segment"])
            bits=[f"traffic share {r['share_normal_pct']}% -> {r['share_anomaly_pct']}%",
                  f"p75 {r['p75_normal']}ms -> {r['p75_anomaly']}ms"]
            role=("gained share and slowed on its own" if r.get("gained_share") and r.get("self_regressed")
                  else "grew its share of traffic" if r.get("gained_share")
                  else "became slower on its own")
            lines.append(f"- {tok} ({role}): " + " , ".join(bits))
        extra=f["segments"].get("additional_count", 0)
        if extra:
            lines.append(f"- Plus {extra} more page section(s) with smaller contributions.")
    else:
        for mv in f["segments"]["primary_share_movers"][:3]:
            lines.append(f"- {from_token(mv['segment'])}: traffic share "
                         f"{mv['share_normal_pct']}% -> {mv['share_anomaly_pct']}% , "
                         f"p75 {mv['p75_normal']}ms -> {mv['p75_anomaly']}ms")
    b=f.get("behavior", {})
    if b.get("new_visitor_influx"):
        lines.append(f"- Audience indicator (new visitors, not a cause): session-entry "
                     f"share {b['normal']['landing_share_pct']}% -> {b['anomaly']['landing_share_pct']}%, "
                     f"browser cache hit median {b['normal']['client_cacherate_median']} -> "
                     f"{b['anomaly']['client_cacherate_median']}")
    cov=f.get("coverage")
    if cov and not cov.get("sufficient"):
        lines.append(f"- Coverage note: the sections above explain "
                     f"{int(round(cov['coverage_ratio']*100))}% of the p75 change "
                     f"({cov['explained_p75_delta']}ms of {cov['overall_p75_delta']}ms); "
                     f"a further contributor remains unaccounted for.")
    ob=f.get("other_watch")
    if ob and ob.get("flagged"):
        lines.append(f"- The catch-all 'other' bucket also degraded "
                     f"(p75 +{ob['p75_delta_ms']}ms); a page section beyond the tracked "
                     f"set may be involved.")
    lines.append("")
    lines.append("## What Did Not Change")
    if f["delivery"]["verdict"]=="clean":
        lines.append("- CDN and origin delivery metrics show no regression.")
    if f.get("localization", {}).get("localized"):
        ex=f["localization"]["excluded_p75"]
        lines.append(f"- Excluding the focus section(s), sitewide p75 moved "
                     f"{ex[0]}ms -> {ex[1]}ms.")
    lines.append("")
    lines.append("## Recommended Actions")
    actions={
        "traffic_mix_shift":[
            "Pre-optimize the growing page section's largest visual element (preload, right-sized images) for first-time mobile visitors.",
            "Apply adaptive image/video delivery for cellular connections in the growing regions.",
            "Split alerting by page group and region during campaign periods to avoid composition-driven alerts."],
        "mix_shift_with_local_regression":[
            "Investigate the growing section's own slowdown: compare the LCP element and resource waterfall between the two windows.",
            "Pre-optimize that section's hero image/video (preload, right-sized, adaptive for cellular) for cold-cache first-time visitors.",
            "Verify no recent release or third-party tag change landed on that section in the anomaly window.",
            "Split alerting by page group and region so the composition shift and the local regression are tracked separately."],
        "multi_segment_mix_shift":[
            "Address each listed section's traffic growth: right-size and preload its main visual element for new mobile visitors.",
            "Prioritize the sections by their contribution score shown above.",
            "Split alerting by page group so simultaneous composition shifts are visible individually."],
        "multi_segment_regression":[
            "Treat each listed section as a separate regression: compare LCP element and resource waterfalls per section between windows.",
            "Check for a shared root cause across the affected sections (common template, shared third-party tag, platform release).",
            "Prioritize remediation by the contribution score shown above.",
            "Split alerting by page group so concurrent regressions are not averaged away."],
        "traffic_mix_shift_broad":[
            "Review campaign traffic routing and landing-page weight across the growing segments.",
            "Split alerting by page group and region."],
        "delivery_regression":[
            "Investigate CDN cache hit rate and origin response times for the anomaly window.",
            "Check recent configuration or deployment changes on the delivery path."],
        "segment_regression":[
            "Debug the slowed segments individually (page weight, third-party tags, recent releases).",
            "Compare resource waterfalls between windows for the affected segments."],
        "no_action":[
            "No action required; continue monitoring.",
            "Consider percentile-based alerting to reduce sensitivity to extreme-duration beacons."],
    }
    for a in actions.get(v["code"], actions["no_action"]):
        lines.append(f"- {a}")
    if cov and not cov.get("sufficient"):
        lines.append("- Widen the tracked page-group set (increase top_k) or analyze another dimension; the current sections do not fully explain the change.")
    lines.append("")
    lines.append("## Monitoring Notes")
    lines.append("Watch whether the p75 returns to baseline as the traffic composition normalizes.")
    return "\n".join(lines)


# ------------------------------------------------------------- email html
def md_to_html(text):
    lines, html, in_list = text.splitlines(), [], False
    for line in lines:
        if line.startswith("- "):
            if not in_list:
                html.append("<ul>")
                in_list = True
            html.append(f"<li>{line[2:]}</li>")
            continue
        if in_list:
            html.append("</ul>")
            in_list = False
        if line.startswith("### "):
            html.append(f"<h3>{line[4:]}</h3>")
        elif line.startswith("## "):
            html.append(f"<h2>{line[3:]}</h2>")
        elif line.startswith("# "):
            html.append(f"<h1>{line[2:]}</h1>")
        elif line.strip() == "":
            html.append("")
        else:
            line = re.sub(r"\*\*(.+?)\*\*", r"<strong>\1</strong>", line)
            html.append(f"<p>{line}</p>")
    if in_list:
        html.append("</ul>")
    return ("<html><body style='font-family:Arial,sans-serif;max-width:720px'>"
            + "\n".join(html) + "</body></html>")


# ============================ v3 additions ============================
PG_TOKEN = "\u27e6PG:{}\u27e7"          # ⟦PG:unpacked⟧ — LLM-opaque placeholder

def page_token(segment):
    return PG_TOKEN.format(segment)

def render_page_tokens(text, url_map, label_map):
    """Deterministically replace ⟦PG:<seg>⟧ tokens with a customer-readable
    label + a concrete representative URL on FIRST mention, label-only after.
    This does NOT depend on the LLM reproducing any exact phrase, so URLs can
    never be silently dropped or mangled."""
    seen = set()
    def _repl(m):
        seg = m.group(1)
        label = label_map.get(seg, f"the '{seg}' page section")
        url = url_map.get(seg)
        if seg in seen or not url:
            return label
        seen.add(seg)
        return f"{label} (e.g., {url})"
    out = _re.sub(r"\u27e6PG:([^\u27e7]+)\u27e7", _repl, text)
    # tidy any accidental double article the LLM may have written before a token
    out = _re.sub(r"\b(the|The)\s+the\b", r"\1", out)
    return out


CAUSAL_WORDS = ("driver", "drivers", "cause", "caused", "causes", "root cause",
                "responsible for", "led to", "leading to", "due to")

def check_causal_misuse(text, symptom_labels):
    """Flag drafts that describe an ASSOCIATED SYMPTOM (e.g. browser cache hit
    rate) as a causal driver. Returns the offending symptom labels."""
    offenders = []
    low = text.lower()
    for lbl in symptom_labels:
        l = lbl.lower()
        if l not in low:
            continue
        for m in _re.finditer(_re.escape(l), low):
            window = low[max(0, m.start() - 60): m.end() + 60]
            if any(w in window for w in CAUSAL_WORDS):
                offenders.append(lbl)
                break
    return sorted(set(offenders))


# ============================ v6 validation layer ============================
# Fixes the deadlock and false-positive problems found in v5 operation:
#   - numbers we ourselves put in the prompt were never whitelisted
#   - a single soft violation discarded an otherwise-good draft
#   - the causal gate matched across sentence boundaries

STRICT_SECTIONS = ("executive summary", "what changed", "what did not change")

def build_allowed_numbers(findings, findings_text, extra=()):
    """v6: whitelist = numeric values in findings  UNION  every number that
    literally appears in the prompt text we hand the model (playbook actions,
    lever names, hypotheses...). Without the second half the model is punished
    for faithfully quoting our own strings."""
    allowed = set(collect_numbers(findings))
    for m in re.finditer(r"(?<![\w.])\d[\d,]*(?:\.\d+)?", findings_text or ""):
        try:
            allowed.add(round(float(m.group(0).replace(",", "")), 2))
        except ValueError:
            pass
    for v in extra:
        try:
            allowed.add(round(float(v), 2))
        except (TypeError, ValueError):
            pass
    return allowed


def _split_sections(text):
    """Return [(section_title_lower, body), ...] for a markdown report."""
    parts = re.split(r"^##\s*(.+)$", text, flags=re.MULTILINE)
    out, i = [], 1
    if parts and parts[0].strip():
        out.append(("", parts[0]))
    while i + 1 < len(parts) + 1 and i < len(parts):
        title = parts[i].strip().lower()
        body = parts[i + 1] if i + 1 < len(parts) else ""
        out.append((title, body))
        i += 2
    return out


def validate_report_numbers_v6(report_text, allowed, small_int_max=15):
    """Section-aware number validation. Numbers inside claim sections are HARD
    violations; numbers elsewhere (recommendations, monitoring, hypotheses —
    where product names and protocol numbers legitimately appear) are SOFT and
    can be auto-repaired instead of triggering a full regeneration."""
    hard, soft = [], []
    for title, body in _split_sections(report_text):
        body_wo_urls = re.sub(r"https?://\S+", "", body)
        strict = any(s in title for s in STRICT_SECTIONS)
        for m in re.finditer(r"(?<![\w.])\d[\d,]*(?:\.\d+)?", body_wo_urls):
            raw = m.group(0).replace(",", "")
            try:
                v = float(raw)
            except ValueError:
                continue
            if v <= small_int_max and v == int(v):
                continue
            if any(abs(v - a) <= 0.51 for a in allowed):
                continue
            (hard if strict else soft).append(m.group(0))
    return {"hard": sorted(set(hard)), "soft": sorted(set(soft))}


# ---------------------------------------------------------- causal gate v6
CAUSAL_PATTERNS = [
    r"\bis (?:the|a|one of the)?\s*(?:most significant |main |primary |key |top )?"
    r"(?:driver|cause|root cause|reason)\b",
    r"\bwas (?:the|a)\s*(?:driver|cause|root cause|reason)\b",
    r"\bdrove\b", r"\bcaused\b", r"\bled to\b", r"\bresponsible for\b",
    r"\bmade the (?:site|page|experience) slow", r"\bstemmed from\b",
    r"\bthe culprit\b", r"\bis what (?:made|caused)\b",
]
NEGATION_MARKERS = ("not a cause", "not the cause", "not a driver",
                    "not the driver", "rather than a cause", "not because",
                    "is not responsible", "indicator", "symptom",
                    "not a performance cause")


def _sentences(text):
    return re.split(r"(?<=[.!?])\s+|\n", text)


def check_causal_misuse_v6(text, symptom_labels):
    """v6: sentence-scoped and syntactically bound. A symptom is flagged only
    when a causal construction appears IN THE SAME SENTENCE and that sentence
    is not explicitly disclaiming causality. Deliberately conservative — the
    critic pass handles subtler cases."""
    offenders = []
    for sent in _sentences(text):
        low = sent.lower()
        if not low.strip():
            continue
        if any(n in low for n in NEGATION_MARKERS):
            continue
        if not any(re.search(p, low) for p in CAUSAL_PATTERNS):
            continue
        for lbl in symptom_labels:
            if lbl.lower() in low:
                offenders.append(lbl)
    return sorted(set(offenders))


# ------------------------------------------------------- repair & scoring
def repair_soft_numbers(report_text, soft_numbers):
    """Deterministically neutralize soft violations: drop the offending
    numeric token (and a bare parenthetical wrapper if that's all it held)
    rather than discarding an otherwise-valid draft."""
    out = report_text
    units = r"(?:\s*(?:ms|s|kb|mb|gb|%|px|KB|MB|GB|MS))?"
    for num in soft_numbers:
        out = re.sub(r"\s*\(\s*" + re.escape(num) + units + r"\s*\)", "", out)
        out = re.sub(r"(?<![\w.])" + re.escape(num) + units + r"(?![\w.])", "", out)
    out = re.sub(r"\(\s*[/,;]?\s*\)", "", out)      # empty parens left behind
    out = re.sub(r"[ \t]{2,}", " ", out)
    out = re.sub(r"\s+([.,;])", r"\1", out)
    return out


def score_draft(num_result, raw_feats, causal_bad, scope_bad, has_summary):
    """Lower is better. Hard problems weigh far more than soft ones so a
    near-miss draft still beats the generic template."""
    return (100 * len(num_result["hard"]) + 100 * len(raw_feats)
            + 60 * len(causal_bad) + 60 * len(scope_bad)
            + 5 * len(num_result["soft"]) + (0 if has_summary else 200))


# ========================== v6.1: draft normalization ==========================
def normalize_draft(text):
    """Absorb harmless formatting variance instead of rejecting it. Strips code
    fences and <think> blocks (Qwen3 emits these), and normalizes any heading
    style to '## Title'. Heading spelling is unrelated to factual accuracy, so
    tolerating it saves regeneration cycles on smaller models."""
    t = text.strip()
    t = re.sub(r"<think>.*?</think>", "", t, flags=re.DOTALL | re.IGNORECASE)
    t = re.sub(r"^\s*```[a-zA-Z]*\s*\n?", "", t)
    t = re.sub(r"\n?```\s*$", "", t)
    t = re.sub(r"^\s*\*\*(.{3,60}?)\*\*\s*:?\s*$", r"## \1", t, flags=re.M)
    t = re.sub(r"^\s{0,3}#{1,6}\s*\d+[.)]\s*", "## ", t, flags=re.M)
    t = re.sub(r"^\s{0,3}#{1,6}\s*", "## ", t, flags=re.M)
    return t.strip()


REQUIRED_SECTION = r"^##\s*executive\s+summary"

def has_required_structure(text):
    return bool(re.search(REQUIRED_SECTION, text, re.M | re.I))


# ===================== v6.1: year = hard violation anywhere =====================
def validate_report_numbers_v61(report_text, allowed, small_int_max=15):
    """As v6, plus: a bare 4-digit year (19xx/20xx) is never a legitimate metric
    in this report, so it is a HARD violation regardless of section."""
    res = validate_report_numbers_v6(report_text, allowed, small_int_max)
    promoted = [n for n in res["soft"] if re.fullmatch(r"(19|20)\d{2}", n.replace(",", ""))]
    if promoted:
        res = {"hard": sorted(set(res["hard"]) | set(promoted)),
               "soft": [n for n in res["soft"] if n not in promoted]}
    return res


# ================== v6.1: model-independent example-URL injection ==================
def inject_segment_urls(text, url_map, label_map, focus_segments=None):
    """Guarantees every focus section carries a concrete example URL WITHOUT
    depending on the model emitting anything. Three stages:
      1. resolve ⟦PG:seg⟧ tokens (if the model cooperated),
      2. otherwise annotate the first plain-prose mention of the segment name
         (quoted / hyphen / underscore / case variants),
      3. append a 'Page Sections Referenced' block for anything still missing.
    """
    out = text

    def _tok(m):
        seg = m.group(1)
        lbl = label_map.get(seg, f"the '{seg}' page section")
        url = url_map.get(seg)
        return f"{lbl} (e.g., {url})" if url else lbl
    out = re.sub(r"\u27e6PG:([^\u27e7]+)\u27e7", _tok, out)

    annotated = {s for s, u in url_map.items() if u and u in out}
    # Stage 2 is restricted to the FOCUS sections and requires the mention to
    # look like a section reference (quoted, or next to section/page wording).
    # Without this a group named e.g. "event" would match the ordinary English
    # word in "campaign or event launch".
    ctx = r"(?:page\s+section|section|page|pages|area|group)"
    for seg in (focus_segments or []):
        url = url_map.get(seg)
        if not url or seg in annotated:
            continue
        variants = sorted({seg, seg.replace("-", " "), seg.replace("_", " ")},
                          key=len, reverse=True)
        name = "(?:" + "|".join(re.escape(v) for v in variants) + ")"
        candidates = [
            r"[\"'\u201c]" + name + r"[\"'\u201d]",            # quoted mention
            name + r"\s+" + ctx,                                # "unpacked section"
            ctx + r"\s+" + name,                                # "section unpacked"
        ]
        for pat in candidates:
            m = re.search(r"(?<![\w/-])" + pat + r"(?![\w/-])", out, re.IGNORECASE)
            if m:
                out = out[:m.end()] + f" (e.g., {url})" + out[m.end():]
                annotated.add(seg)
                break

    missing = [s for s in (focus_segments or [])
               if s not in annotated and url_map.get(s)]
    if missing:
        block = ["", "## Page Sections Referenced"]
        block += [f"- {label_map.get(s, s)}: {url_map[s]}" for s in missing]
        out = out.rstrip() + "\n" + "\n".join(block)
    return out


def verify_urls_present(text, url_map, focus_segments):
    """Final assertion: which focus sections still lack their example URL."""
    return [s for s in focus_segments if url_map.get(s) and url_map[s] not in text]


# ================== v6.1: number-context binding (misassignment) ==================
def build_metric_bindings(findings):
    """Map a metric keyword to the set of values legitimately associated with it.
    Catches the failure mode a plain whitelist cannot: a number that EXISTS in
    findings but is attached to the wrong metric (e.g. quoting the mean-based
    decomposition total as the p75 change)."""
    h = findings.get("headline", {})
    b = findings.get("behavior", {}) or {}
    d = findings.get("decomposition", {}) or {}
    bind = {}
    p75_vals = {h.get("p75_normal"), h.get("p75_anomaly"),
                h.get("delta_ms"), h.get("delta_pct")}
    for r in (findings.get("segments", {}).get("focus_list") or []):
        p75_vals |= {r.get("p75_normal"), r.get("p75_anomaly"), r.get("p75_delta_ms")}
    for ci in (h.get("ci95") or []):
        p75_vals.add(ci)
    bind["p75"] = {round(float(v), 2) for v in p75_vals if isinstance(v, (int, float))}

    dec_vals = {d.get("total_delta_ms"), d.get("mix_effect_ms"), d.get("within_effect_ms")}
    bind["mix effect"] = bind["composition"] = {
        round(float(v), 2) for v in dec_vals if isinstance(v, (int, float))}

    land = {(b.get("normal") or {}).get("landing_share_pct"),
            (b.get("anomaly") or {}).get("landing_share_pct")}
    bind["session-entry"] = bind["landing"] = {
        round(float(v), 2) for v in land if isinstance(v, (int, float))}

    cache = {(b.get("normal") or {}).get("client_cacherate_median"),
             (b.get("anomaly") or {}).get("client_cacherate_median")}
    bind["cache"] = {round(float(v), 2) for v in cache if isinstance(v, (int, float))}
    return {k: v for k, v in bind.items() if v}


def check_number_binding(report_text, bindings, small_int_max=15):
    """Flag sentences where a metric keyword co-occurs with a number that is not
    in that metric's legitimate value set."""
    issues = []
    for sent in re.split(r"(?<=[.!?])\s+|\n", report_text):
        low = sent.lower()
        if not low.strip():
            continue
        clean = re.sub(r"https?://\S+", "", sent)
        for key, allowed_vals in bindings.items():
            if key not in low:
                continue
            for m in re.finditer(r"(?<![\w.])\d[\d,]*(?:\.\d+)?", clean):
                try:
                    v = float(m.group(0).replace(",", ""))
                except ValueError:
                    continue
                if v <= small_int_max and v == int(v):
                    continue
                if not any(abs(v - a) <= 0.51 for a in allowed_vals):
                    issues.append(f"'{m.group(0)}' used with '{key}'")
            break
    return sorted(set(issues))


# ===================== v6.1: pre-rendered fact sentences =====================
def build_narrative_facts(findings):
    """Python writes the sentences for every load-bearing figure; the model is
    told to reuse them. This removes the need for the model to pick the right
    field out of a large JSON — the root cause of number misassignment."""
    facts = {}
    h = findings.get("headline", {})
    if h.get("transition_sentence"):
        facts["headline"] = h["transition_sentence"]
    d = findings.get("decomposition") or {}
    if d.get("total_delta_ms") is not None:
        facts["decomposition"] = (
            f"Splitting the {d.get('stat','mean')} change of {d['total_delta_ms']}ms, "
            f"{d.get('mix_effect_ms')}ms comes from traffic composition and "
            f"{d.get('within_effect_ms')}ms from sections changing their own speed.")
    b = findings.get("behavior") or {}
    n, a = b.get("normal") or {}, b.get("anomaly") or {}
    if n.get("landing_share_pct") is not None and a.get("landing_share_pct") is not None:
        facts["audience"] = (
            f"Session-entry page views moved from {n['landing_share_pct']}% to "
            f"{a['landing_share_pct']}%, and the median browser cache hit rate from "
            f"{n.get('client_cacherate_median')} to {a.get('client_cacherate_median')} "
            f"— indicators of more first-time visitors, not causes of the slowdown.")
    for r in (findings.get("segments", {}).get("focus_list") or []):
        facts[f"section::{r['segment']}"] = (
            f"{page_token(r['segment'])} moved from {r['share_normal_pct']}% to "
            f"{r['share_anomaly_pct']}% of traffic, with its own p75 going from "
            f"{r['p75_normal']}ms to {r['p75_anomaly']}ms.")
    c = findings.get("coverage") or {}
    if c.get("coverage_ratio") is not None:
        facts["coverage"] = (
            f"The listed sections account for {c['explained_p75_delta']}ms of the "
            f"{c['overall_p75_delta']}ms p75 change.")
    return facts


def check_degenerate_comparison(report_text, tol=0.01):
    """Catch 'X vs X' / 'from X to X' style claims where both sides are the same
    value — a comparison that carries no information and almost always means the
    model pulled the same field twice (e.g. '79.8% vs 79.8%')."""
    issues = []
    pats = [r"(\d[\d,]*(?:\.\d+)?)\s*%?\s*(?:vs\.?|versus|compared to)\s*(\d[\d,]*(?:\.\d+)?)\s*%?",
            r"from\s+(\d[\d,]*(?:\.\d+)?)\s*%?\s*(?:ms)?\s*to\s+(\d[\d,]*(?:\.\d+)?)\s*%?"]
    for p in pats:
        for m in re.finditer(p, report_text, re.I):
            try:
                a = float(m.group(1).replace(",", "")); b = float(m.group(2).replace(",", ""))
            except ValueError:
                continue
            if abs(a - b) <= tol:
                issues.append(f"degenerate comparison '{m.group(0).strip()}'")
    return sorted(set(issues))


In [ ]:
# Cell 4b — LLM-enhancement layer: Akamai playbook, hypotheses, critic, scope guard (v6.1)
"""v5 LLM-enhancement layer.

Adds, on top of the deterministic findings pipeline:
  1. Playbook-grounded recommendations — Akamai-lever remediation actions
     selected by matching findings signals to a structured playbook, so the
     LLM specifies concrete, in-scope actions instead of generic advice.
  2. Critic pass — a second LLM call that audits the draft against the
     findings (numbers, symptom/cause discipline, verdict alignment,
     out-of-scope recommendations) and returns a structured verdict.
  3. Hypothesis layer — findings-consistent external explanations, always
     confined to a clearly-labeled "Hypotheses (to verify)" section and
     never mixed with established facts.

Everything is generic: the playbook is data, not code, and matching is by
signal predicates evaluated against the findings dict.
"""
import json
import re


# ============================================================= PLAYBOOK
# Akamai-lever remediation playbook (draft). Each entry:
#   id, when: list of signal predicates (ALL must hold), levers, action (LLM
#   rewrites into customer prose), scope_tag (used by the scope guard).
# Signals are simple dotted-path + operator checks against findings.
AKAMAI_PLAYBOOK = [
    {
        "id": "ivm_hero_cold_cache",
        "when": ["verdict in traffic_mix_shift,mix_shift_with_local_regression,"
                 "multi_segment_mix_shift,multi_segment_regression",
                 "behavior.new_visitor_influx == true"],
        "levers": ["Image & Video Manager (adaptive/right-sized hero media)",
                   "EdgeWorkers (LCP element preload hint injection)"],
        "action": ("Right-size and preload the growing section's LCP element "
                   "(hero image/video) for cold-cache first-time visitors, and "
                   "serve adaptive quality on cellular connections."),
        "scope_tag": "edge_media",
    },
    {
        "id": "prefetch_event_landing",
        "when": ["behavior.new_visitor_influx == true",
                 "behavior.anomaly.landing_share_pct > 70"],
        "levers": ["EdgeWorkers (Early Hints)",
                   "Adaptive Acceleration (automatic push/preload)"],
        "action": ("Enable early-hints/preload for the critical LCP and font "
                   "resources on the campaign landing pages so first paint is "
                   "not blocked for direct-entry visitors."),
        "scope_tag": "edge_hints",
    },
    {
        "id": "offload_regional_origin",
        "when": ["delivery.verdict == clean",
                 "within_regression.within_regression == true"],
        "levers": ["Tiered Distribution / cache key review",
                   "Cloud Wrapper (origin offload for cold regions)"],
        "action": ("Confirm the growing region is served from a nearby edge "
                   "tier and review the cache key so first-time visitors in "
                   "that region still benefit from a warm shared cache."),
        "scope_tag": "edge_cache",
    },
    {
        "id": "delivery_investigate",
        "when": ["delivery.verdict == degraded"],
        "levers": ["mPulse + DataStream 2 correlation",
                   "Origin health / offload review"],
        "action": ("Investigate the delivery-path regression: correlate the "
                   "affected window in DataStream 2 with origin response times "
                   "and cache offload before any content change."),
        "scope_tag": "delivery_ops",
    },
    {
        "id": "alert_segmentation",
        "when": ["verdict in traffic_mix_shift,multi_segment_mix_shift,"
                 "mix_shift_with_local_regression,multi_segment_regression"],
        "levers": ["mPulse alert configuration (page-group / geo dimensions)"],
        "action": ("Split mPulse alerting by page group and region during "
                   "campaign periods so composition shifts and genuine "
                   "regressions are tracked separately, reducing false alarms."),
        "scope_tag": "monitoring",
    },
    {
        "id": "third_party_release_audit",
        "when": ["within_regression.within_regression == true"],
        "levers": ["Script Management / third-party tag review",
                   "release-change correlation"],
        "action": ("Verify no recent release or third-party tag change landed "
                   "on the affected section in the anomaly window; compare the "
                   "resource waterfall between the two windows."),
        "scope_tag": "app_change",
    },
]

VALID_SCOPE_TAGS = {p["scope_tag"] for p in AKAMAI_PLAYBOOK}


def _get_path(d, path):
    cur = d
    for part in path.split("."):
        if isinstance(cur, dict) and part in cur:
            cur = cur[part]
        else:
            return None
    return cur


def _eval_predicate(findings, pred):
    """Evaluate one predicate string against findings. Supports:
       'a.b == x', 'a.b > n', 'a.b < n', 'a.b in x,y,z'."""
    m = re.match(r"^(.+?)\s+(==|>|<|in)\s+(.+)$", pred.strip())
    if not m:
        return False
    path, op, rhs = m.group(1), m.group(2), m.group(3).strip()
    val = _get_path(findings, path.strip())
    if op == "in":
        opts = [x.strip() for x in rhs.split(",")]
        return str(val) in opts
    if op == "==":
        if rhs in ("true", "false"):
            return bool(val) == (rhs == "true")
        return str(val) == rhs
    try:
        num = float(rhs); val = float(val)
    except (TypeError, ValueError):
        return False
    return val > num if op == ">" else val < num


def match_playbook(findings, playbook=AKAMAI_PLAYBOOK):
    """Return the remediation entries whose predicates all hold, with the
    verdict code exposed at top level for the 'verdict in ...' predicates."""
    ctx = dict(findings)
    ctx["verdict"] = findings.get("verdict", {}).get("code")
    selected = []
    for entry in playbook:
        if all(_eval_predicate(ctx, p) for p in entry["when"]):
            selected.append({"id": entry["id"], "levers": entry["levers"],
                             "action": entry["action"],
                             "scope_tag": entry["scope_tag"]})
    return selected


# ============================================================= HYPOTHESES
def derive_hypotheses(findings):
    """Findings-consistent external explanations. Deterministic seeds the LLM
    may phrase; each is explicitly a hypothesis, never asserted as fact."""
    hyps = []
    beh = findings.get("behavior", {})
    if beh.get("new_visitor_influx"):
        fl = findings.get("segments", {}).get("focus_list") or []
        where = ""
        dd = findings.get("segments", {}).get("drilldown", {}).get("country")
        if dd:
            gainers = [r for r in dd if r.get("share_delta_pp", 0) > 1]
            if gainers:
                where = " concentrated in " + ", ".join(
                    r["segment"] for r in gainers[:2])
        hyps.append("A marketing campaign or product-announcement event drove "
                    "a burst of first-time visitors" + where + ", which arrive "
                    "with empty browser caches and therefore slower first loads.")
    if findings.get("within_regression", {}).get("within_regression"):
        hyps.append("A content or third-party change may have shipped to the "
                    "affected section shortly before the window, adding to its "
                    "load time independently of the traffic shift.")
    cov = findings.get("coverage")
    if cov and not cov.get("sufficient"):
        hyps.append("Part of the change originates outside the identified "
                    "sections; a broader release or a page group beyond the "
                    "tracked set may contribute.")
    return hyps


# ============================================================= CRITIC
def build_critic_prompt(report_text, findings_text):
    return f"""You are a strict reviewer auditing a performance report draft
against its source findings. Return ONLY a JSON object, no prose.

Check for these problems:
- "invented_numbers": any number in the report not present in FINDINGS.
- "symptom_as_cause": any associated symptom (e.g. cache hit rate, bytes,
  session-entry share) described as a driver/cause/root cause.
- "verdict_conflict": any statement contradicting findings.verdict.sentence.
- "unsupported_causal_claim": a causal claim not backed by findings.
- "missing_focus_section": a page section in findings.segments.focus_list not
  mentioned in the report.

Return exactly:
{{"pass": true|false,
  "issues": [{{"type": "<one of the above>", "detail": "<short quote or note>"}}],
  "severity": "none|minor|major"}}

FINDINGS:
{findings_text}

REPORT DRAFT:
{report_text}
"""


def parse_critic_response(text):
    """Robustly extract the critic JSON."""
    m = re.search(r"\{.*\}", text, re.DOTALL)
    if not m:
        return {"pass": True, "issues": [], "severity": "none",
                "_parse_error": True}
    try:
        obj = json.loads(m.group(0))
        obj.setdefault("pass", not obj.get("issues"))
        obj.setdefault("issues", [])
        obj.setdefault("severity", "none")
        return obj
    except json.JSONDecodeError:
        return {"pass": True, "issues": [], "severity": "none",
                "_parse_error": True}


# ============================================================= SCOPE GUARD
def check_recommendation_scope(report_text, allowed_playbook_actions):
    """Deterministic guard: the Recommended Actions section should not stray
    into levers outside the matched playbook. We check that the section does
    not introduce recommendation verbs about systems we never selected.
    Returns a list of out-of-scope hints (advisory)."""
    # extract the actions section
    m = re.search(r"##\s*Recommended Actions(.*?)(?:\n##|\Z)", report_text,
                  re.DOTALL | re.IGNORECASE)
    if not m:
        return []
    section = m.group(1).lower()
    # systems that must only appear if a matching playbook entry was selected
    guarded = {
        "waf": "security", "rate limit": "security", "captcha": "security",
        "database": "backend", "sql": "backend",
        "autoscal": "backend", "kubernetes": "backend",
    }
    offenders = []
    for term, area in guarded.items():
        if term in section:
            offenders.append(f"out-of-scope ({area}): '{term}'")
    return offenders


In [ ]:
# Cell 4c — v5 fallback extension: playbook-grounded actions + hypotheses in the deterministic report
def render_fallback_report_v5(f):
    base = render_fallback_report(f)  # emits tokens; resolved by caller
    # replace the generic Recommended Actions block with playbook actions when present
    pb = f.get("remediation_playbook") or []
    if pb:
        lines = base.split("\n")
        out, i = [], 0
        while i < len(lines):
            if lines[i].strip().lower().startswith("## recommended actions"):
                out.append("## Recommended Actions")
                for e in pb:
                    levers = "; ".join(e["levers"])
                    out.append(f"- {e['action']} (Akamai: {levers})")
                # skip original action bullets until next section
                i += 1
                while i < len(lines) and not lines[i].startswith("## "):
                    i += 1
                continue
            out.append(lines[i]); i += 1
        base = "\n".join(out)
    # append hypotheses section
    hyps = f.get("hypotheses") or []
    if hyps:
        base += "\n\n## Hypotheses (to verify)\n" + "\n".join(f"- {h}" for h in hyps)
    return base


In [ ]:
# Cell 5 — Data load + sample-integrity gate
import json, warnings
import pandas as pd
warnings.filterwarnings("ignore")

csv_path = PROCESSED_DIR / CSV_FILENAME
if not csv_path.is_file():
    csv_path = Path(CSV_FILENAME)          # notebook-test fallback: local file
df = pd.read_csv(csv_path)
print(f"loaded {len(df):,} rows from {csv_path}")

# ---- ground truth: prefer upstream sidecar metadata over the (possibly sampled) CSV
source_meta = None
if METADATA_PATH.is_file():
    source_meta = json.loads(METADATA_PATH.read_text())
    print("sidecar metadata found — report numbers will use SOURCE stats")
    for lab, key in [(0, "normal"), (1, "anomaly")]:
        sp = float(df.loc[df[LABEL_COL]==lab, TIMER_COL].quantile(.75))
        gt = float(source_meta["windows"][key]["p75"])
        drift = abs(sp - gt) / gt * 100
        print(f"  {key}: sample p75={sp:.1f} vs source p75={gt:.1f} (drift {drift:.2f}%)")
        if drift > SAMPLE_DRIFT_TOL:
            raise RuntimeError(
                f"Sample does not represent source data for '{key}' window "
                f"({drift:.1f}% p75 drift > {SAMPLE_DRIFT_TOL}%). "
                "Fix upstream sampling (use stratified label x timer-decile sampling) before reporting.")
else:
    print("[warn] no sidecar metadata — falling back to CSV-computed stats. "
          "Upstream should emit window counts + percentiles at export time.")

# ---- derived, dataset-agnostic features
df["page_group"] = derive_page_group(df["url"]) if "url" in df else "all"
df["referrer_present"] = df["referrer"].notna() if "referrer" in df else False

# ---- representative URL per page group, so the report can cite a concrete
#      example page instead of only an abstract group name.
def representative_urls(frame, group_col="page_group", url_col="url",
                        skip=("other", "unknown", "all")):
    """Most frequent concrete URL (query string stripped) per page group."""
    mapping = {}
    if url_col not in frame:
        return mapping
    for grp, sub in frame.groupby(group_col):
        if str(grp) in skip:
            continue
        urls = sub[url_col].dropna().astype(str).map(lambda u: u.split("?")[0])
        if urls.empty:
            continue
        mapping[str(grp)] = urls.value_counts().index[0]
    return mapping

PAGE_GROUP_URLS = representative_urls(df)
print(f"representative URLs mapped for {len(PAGE_GROUP_URLS)} page groups")

In [ ]:
# Cell 6 — Step A: window statistics + severity gate
win = compute_window_stats(df, TIMER_COL, LABEL_COL)
if source_meta:   # override headline percentiles with source ground truth
    for key in ("normal", "anomaly"):
        win[key].update({k: source_meta["windows"][key][k]
                         for k in ("count","mean","p50","p75","p90","p95","p99")
                         if k in source_meta["windows"][key]})
    win["delta_p75_ms"]  = round(win["anomaly"]["p75"] - win["normal"]["p75"], 2)
    win["delta_p75_pct"] = round(win["delta_p75_ms"] / win["normal"]["p75"] * 100, 2)

severity, severity_msg = classify_severity(win, abs_floor_ms=SEVERITY_FLOOR_MS)
print(f"p75: {win['normal']['p75']} -> {win['anomaly']['p75']} "
      f"({win['delta_p75_ms']:+}ms, {win['delta_p75_pct']:+}%)  "
      f"CI95={win['delta_p75_ci95']}  MW-p={win['mannwhitney_p']:.2e}")
print(f"severity: {severity} — {severity_msg}")

In [ ]:
# Cell 7 — Step B: artifact/outlier audit
outliers = audit_outliers(df, TIMER_COL, LABEL_COL, ARTIFACT_MS)
for k, v in outliers["windows"].items():
    print(f"{k:>7}: {v['artifact_count']} beacons >{ARTIFACT_MS/1000:.0f}s "
          f"({v['artifact_share_pct']}%), mean inflated by {v['mean_inflation_ms']}ms, "
          f"max={v['max_timer_ms']:,}ms")

In [ ]:
# Cell 8 — Step C: decomposition, mover discovery, MULTI-focus selection (v4)
decomp={}
for d in [PRIMARY_DIM, *SECONDARY_DIMS[:2], [PRIMARY_DIM, SECONDARY_DIMS[0]]]:
    r=mix_within_decomposition(df, d, TIMER_COL, LABEL_COL)
    decomp[r["dim"]]=r
    print(f"[{r['dim']:<22}] Δmean={r['total_delta_ms']:+}ms = "
          f"mix {r['mix_effect_ms']:+} + within {r['within_effect_ms']:+}")

primary_movers=top_movers(df, PRIMARY_DIM, TIMER_COL, LABEL_COL, min_n=MIN_SEG_N)

# v4: select ALL qualifying problem sections, ranked by contribution
focus_selection=select_focus_segments(primary_movers, win, PRIMARY_DIM,
                                      min_share_pp=MIN_SHARE_PP, min_p75_delta=SEVERITY_FLOOR_MS,
                                      max_focus=MAX_FOCUS)
focus_list=focus_selection["focus_list"]
focus=focus_list[0] if focus_list else None          # primary, for localization/behavior drill
focus_segments=[r["segment"] for r in focus_list]

# coverage: do the chosen sections actually explain the sitewide p75 rise?
coverage=coverage_check(df, PRIMARY_DIM, focus_segments, TIMER_COL, LABEL_COL) if focus_segments else None
other_watch=other_bucket_watch(primary_movers)

localization, drilldown = None, {}
focus_df=df
if focus:
    localization=localization_check(df, PRIMARY_DIM, focus["segment"], TIMER_COL, LABEL_COL)
    focus_df=df[df[PRIMARY_DIM].astype(str)==focus["segment"]]
    for d in SECONDARY_DIMS:
        drilldown[d]=top_movers(focus_df, d, TIMER_COL, LABEL_COL, min_n=max(100, MIN_SEG_N//2))

print(f"\nfocus sections ({len(focus_list)}):")
for r in focus_list:
    role=("share+regression" if r["gained_share"] and r["self_regressed"]
          else "share-gain" if r["gained_share"] else "self-regression")
    print(f"   {r['segment']:<22} score={r['contribution_score']:<7} [{role}] "
          f"share {r['share_normal_pct']}->{r['share_anomaly_pct']}%  "
          f"p75 {r['p75_normal']}->{r['p75_anomaly']}")
if focus_selection["additional_count"]:
    print(f"   (+{focus_selection['additional_count']} more: {focus_selection['additional_segments']})")
if coverage:
    print(f"coverage: focus explains {int(round(coverage['coverage_ratio']*100))}% of the "
          f"p75 change (sufficient={coverage['sufficient']})")
if other_watch["flagged"]:
    print(f"[watch] 'other' bucket degraded: p75 +{other_watch['p75_delta_ms']}ms")

# v4: within-regression is per-focus; compute for the primary focus (kept for compatibility)
within_flags=composite_verdict_flags(focus)
if focus:
    print(f"primary focus within-regression: {within_flags['within_regression']} "
          f"(median Δ={within_flags['focus_median_delta_ms']}ms, p75 Δ={within_flags['focus_p75_delta_ms']}ms)")

In [ ]:
# Cell 9 — Step D+E: audience behavior signals + delivery-layer health
behavior_overall = behavior_signals(df, LABEL_COL)
behavior_focus   = behavior_signals(focus_df, LABEL_COL)
print("new-visitor influx — overall:", behavior_overall["new_visitor_influx"],
      "| focus segment:", behavior_focus["new_visitor_influx"])

delivery = delivery_health(df, LABEL_COL)
print("delivery verdict:", delivery["verdict"], delivery["issues"] or "")
for m, v in delivery["metrics"].items():
    print(f"   {m}: {v['normal_median']} -> {v['anomaly_median']}")

In [ ]:
# Cell 10 — Step F: models as evidence (skipped when nothing to explain)
fingerprint, drivers = None, None
if severity not in ("none", "improved"):
    fingerprint = window_classifier_fingerprint(df, CATEGORICAL, LABEL_COL)
    print(f"composition fingerprint — holdout AUC={fingerprint['holdout_auc']} "
          f"(gate {'passed' if fingerprint['gate_passed'] else 'FAILED — windows statistically similar'})")
    for f_ in fingerprint["fingerprint"][:6]:
        print(f"   {f_['feature']:<38} share {f_['share_normal_pct']}% -> "
              f"{f_['share_anomaly_pct']}%")
    drivers = timer_regressor_drivers(df, CATEGORICAL, DELIVERY_NUMERIC,
                                      TIMER_COL, LABEL_COL, ARTIFACT_MS)
    print(f"\nLCP drivers — regressor R2={drivers['train_r2']}, "
          f"predicted window shift={drivers['predicted_shift_pct']}%")
    for d_ in drivers["drivers"][:8]:
        print(f"   {d_['feature']:<38} {d_['direction']:<9} "
              f"median {d_['value_median_normal']} -> {d_['value_median_anomaly']}")
else:
    print("severity gate: models skipped")

# v3: separate causal driver candidates from associated symptoms (endogenous
# indicators of *who* is visiting, not reasons the site slowed down)
SYMPTOM_FEATURES = {"cacherate", "cdncacherate", "transferbyte", "bodysize",
                    "requestcount", "landingpage", "referrer_present", "paidmedia"}
driver_candidates, associated_symptoms = [], []
if drivers:
    for d_ in drivers["drivers"]:
        base = d_["feature"].split("_")[0]
        target = (associated_symptoms
                  if (d_["feature"] in SYMPTOM_FEATURES or base in SYMPTOM_FEATURES)
                  else driver_candidates)
        target.append(d_)
    print("\ncausal driver candidates:", [d_["feature"] for d_ in driver_candidates][:5])
    print("associated symptoms (NOT causes):", [d_["feature"] for d_ in associated_symptoms][:5])

In [ ]:
# Cell 11 — Step G: findings JSON + verdict (single source of truth)  [v3]
from datetime import datetime, timezone

verdict_code, verdict_sentence = select_verdict(
    severity, decomp[PRIMARY_DIM], localization, behavior_focus, delivery,
    outliers, within_flags=within_flags,
    focus_selection=focus_selection, coverage=coverage)

def _label_movers(movers, dim):
    out = []
    for r in movers:
        r = dict(r)
        r["human_label"] = humanize_feature(f"{dim}_{r['segment']}", FEATURE_LABEL_OVERRIDES)
        r["token"] = page_token(r["segment"])          # ⟦PG:seg⟧ for the LLM to copy
        r["example_url"] = PAGE_GROUP_URLS.get(r["segment"])
        out.append(r)
    return out

# customer-readable label per page group (used by the deterministic token renderer)
PAGE_GROUP_LABELS = {}
for r in primary_movers["share_movers"]:
    PAGE_GROUP_LABELS.setdefault(
        r["segment"], humanize_feature(f"{PRIMARY_DIM}_{r['segment']}", FEATURE_LABEL_OVERRIDES))
for r in focus_list:
    PAGE_GROUP_LABELS[r["segment"]] = humanize_feature(
        f"{PRIMARY_DIM}_{r['segment']}", FEATURE_LABEL_OVERRIDES)

def _label_focus(r):
    r=dict(r)
    r["human_label"]=humanize_feature(f"{PRIMARY_DIM}_{r['segment']}", FEATURE_LABEL_OVERRIDES)
    r["token"]=page_token(r["segment"])
    r["example_url"]=PAGE_GROUP_URLS.get(r["segment"])
    return r

symptom_labels = ([humanize_feature(d_["feature"], FEATURE_LABEL_OVERRIDES)
                   for d_ in associated_symptoms] if drivers else [])

timer_name = "largestcontentfulpaint"
findings = {
    "meta": {"metric": timer_name, "rows": int(len(df)),
             "generated_at": datetime.now(timezone.utc).isoformat(timespec="seconds"),
             "stats_source": "sidecar_metadata" if source_meta else "csv_sample"},
    "headline": {
        "p75_normal": win["normal"]["p75"], "p75_anomaly": win["anomaly"]["p75"],
        "delta_ms": win["delta_p75_ms"], "delta_pct": win["delta_p75_pct"],
        "ci95": win["delta_p75_ci95"], "severity": severity,
        "transition_sentence": (
            f"{timer_name} p75 moved from {win['normal']['p75']}ms (normal window) to "
            f"{win['anomaly']['p75']}ms (anomaly window), a change of "
            f"{win['delta_p75_ms']}ms ({win['delta_p75_pct']}%).")},
    "verdict": {"code": verdict_code, "sentence": verdict_sentence},
    "within_regression": within_flags,
    "segments": {
        "primary_dim": PRIMARY_DIM,
        "primary_share_movers": _label_movers(primary_movers["share_movers"][:5], PRIMARY_DIM),
        "focus_list": [_label_focus(r) for r in focus_list],
        "additional_count": focus_selection["additional_count"],
        "additional_segments": focus_selection["additional_segments"],
        "focus": (dict(focus,
                       human_label=humanize_feature(f"{PRIMARY_DIM}_{focus['segment']}", FEATURE_LABEL_OVERRIDES),
                       token=page_token(focus["segment"]),
                       example_url=PAGE_GROUP_URLS.get(focus["segment"]),
                       reason=focus.get("reason","primary problem section")) if focus else None),
        "drilldown": {d: _label_movers(t["share_movers"][:4], d)
                      for d, t in drilldown.items()}},
    "localization": localization,
    "behavior": behavior_focus | {"scope": (focus["segment"] if focus else "overall")},
    "delivery": delivery,
    "outliers": outliers,
    "decomposition": decomp[PRIMARY_DIM],
    "coverage": coverage,
    "other_watch": other_watch,
    "composition_fingerprint": ([{**f_, "human_label": humanize_feature(
        f_["feature"], FEATURE_LABEL_OVERRIDES)} for f_ in fingerprint["fingerprint"][:6]]
        if fingerprint and fingerprint["gate_passed"] else []),
    "performance_drivers": ([{**d_, "human_label": humanize_feature(
        d_["feature"], FEATURE_LABEL_OVERRIDES)} for d_ in driver_candidates[:6]]
        if drivers else []),
    "associated_symptoms": ([{**d_, "human_label": humanize_feature(
        d_["feature"], FEATURE_LABEL_OVERRIDES),
        "note": "indicator of audience change, not a performance cause"}
        for d_ in associated_symptoms[:6]] if drivers else []),
}
# v5: attach Akamai-playbook remediation and findings-consistent hypotheses
findings["remediation_playbook"] = match_playbook(findings)
findings["hypotheses"] = derive_hypotheses(findings)

# v6.1: Python pre-writes every load-bearing sentence so the model never has to
# pick the right field out of the JSON (root cause of number misassignment)
findings["narrative_facts"] = build_narrative_facts(findings)

allowed_numbers = collect_numbers(findings)
# example URLs contain digits that are references, not metrics — exclude from the number whitelist check
findings_text = json.dumps(findings, indent=2, ensure_ascii=False, default=str)
print(f"verdict={verdict_code} | severity={severity} | "
      f"{len(allowed_numbers)} numbers whitelisted | "
      f"{len(PAGE_GROUP_URLS)} example URLs ready")

In [ ]:
# Cell 12 — Step H: LLM verbalization -> normalize -> validate -> critic (v6.1)
import requests

MAX_ATTEMPTS = 4

def call_llm(prompt, timeout=300, json_mode=False):
    """Backend-agnostic client. Default targets vLLM's OpenAI-compatible API
    (Qwen3-14B-AWQ); set LLM_BACKEND=ollama to use a local Ollama server."""
    if LLM_BACKEND == "ollama":
        r = requests.post(LLM_URL, timeout=timeout, json={
            "model": LLM_MODEL, "prompt": prompt, "stream": False,
            "options": {"temperature": LLM_TEMPERATURE, "top_p": LLM_TOP_P},
            **({"format": "json"} if json_mode else {})})
        r.raise_for_status()
        return r.json().get("response", "")
    payload = {
        "model": LLM_MODEL,
        "messages": [{"role": "user", "content": prompt}],
        "temperature": LLM_TEMPERATURE, "top_p": LLM_TOP_P, "max_tokens": 2048,
        # Qwen3 is a hybrid thinking model; keep reasoning traces out of the draft
        "chat_template_kwargs": {"enable_thinking": LLM_ENABLE_THINKING},
    }
    if json_mode:
        payload["response_format"] = {"type": "json_object"}
    r = requests.post(LLM_URL, timeout=timeout, json=payload,
                      headers={"Authorization": f"Bearer {LLM_API_KEY}"})
    r.raise_for_status()
    return r.json()["choices"][0]["message"]["content"]

def build_llm_prompt_v61(findings_text):
    base = build_llm_prompt(findings_text)
    base = base.replace(
        "## Recommended Actions\n(3-4 practical actions matched to the verdict code.)",
        "## Recommended Actions\n(Use ONLY the actions in findings.remediation_playbook. "
        "Phrase each as a concrete step and name its Akamai lever(s). Do NOT invent "
        "actions or mention systems not in the playbook.)\n"
        "## Hypotheses (to verify)\n(ONLY if findings.hypotheses is non-empty: list them as "
        "clearly-labeled hypotheses to verify, never as established facts. Omit this "
        "section entirely if findings.hypotheses is empty.)")
    base += ("\n\nPRE-WRITTEN FACTS — findings.narrative_facts contains a ready sentence for "
             "every figure that matters. Reuse those sentences verbatim or with minimal "
             "rewording. Do NOT re-derive figures from other parts of the JSON, and never "
             "pair a number with a metric it does not belong to.\n"
             "Output plain Markdown only: no code fences, no reasoning traces, and use the "
             "exact section headings listed above.")
    return base

prompt = build_llm_prompt_v61(findings_text)
allowed_numbers = build_allowed_numbers(findings, findings_text)
metric_bindings = build_metric_bindings(findings)
playbook_actions = findings.get("remediation_playbook", [])
focus_segments = [r["segment"] for r in (findings["segments"].get("focus_list") or [])]

best = {"score": None, "text": None, "num": None}
report_md, report_source = None, "fallback_template"

for attempt in range(1, MAX_ATTEMPTS + 1):
    try:
        cand = call_llm(prompt)
    except Exception as e:
        print(f"[warn] LLM unavailable ({e}); using deterministic fallback")
        break

    # v6.1: absorb formatting variance BEFORE judging content
    cand = normalize_draft(cand)
    cand = inject_segment_urls(cand, PAGE_GROUP_URLS, PAGE_GROUP_LABELS, focus_segments)

    num        = validate_report_numbers_v61(cand, allowed_numbers)
    raw_feat   = [d_["feature"] for d_ in findings["performance_drivers"] if d_["feature"] in cand]
    causal_bad = check_causal_misuse_v6(cand, symptom_labels)
    scope_bad  = check_recommendation_scope(cand, playbook_actions)
    bind_bad   = check_number_binding(cand, metric_bindings)
    degen_bad  = check_degenerate_comparison(cand)
    url_missing= verify_urls_present(cand, PAGE_GROUP_URLS, focus_segments)
    structure  = has_required_structure(cand)

    score = (score_draft(num, raw_feat, causal_bad, scope_bad, structure)
             + 100 * len(bind_bad) + 80 * len(degen_bad) + 50 * len(url_missing))
    if best["score"] is None or score < best["score"]:
        best = {"score": score, "text": cand, "num": num}

    blocking = bool(num["hard"] or raw_feat or causal_bad or scope_bad
                    or bind_bad or degen_bad or url_missing or not structure)
    if not blocking:
        clean = repair_soft_numbers(cand, num["soft"]) if num["soft"] else cand
        try:
            critic = parse_critic_response(
                call_llm(build_critic_prompt(clean, findings_text), json_mode=True))
        except Exception as e:
            critic = {"pass": True, "issues": [], "severity": "none", "_skipped": str(e)}
        if critic.get("pass") or critic.get("severity") in ("none", "minor"):
            report_md, report_source = clean, f"llm+critic ({LLM_MODEL})"
            if critic.get("issues"):
                print(f"[critic] minor issues noted: {critic['issues'][:2]}")
            break
        print(f"[attempt {attempt}] critic rejected (severity={critic.get('severity')}): "
              f"{critic.get('issues', [])[:2]}")
        fixes = "; ".join(i.get("detail", "") for i in critic.get("issues", [])[:3])
    else:
        # v6.1: every blocking condition is now visible, including structure
        print(f"[attempt {attempt}] rejected — hard:{num['hard'][:3]} soft:{num['soft'][:3]} "
              f"raw_feats:{raw_feat[:2]} symptom-as-cause:{causal_bad[:2]} "
              f"out-of-scope:{scope_bad[:2]} number-binding:{bind_bad[:2]} "
              f"degenerate:{degen_bad[:2]} url_missing:{url_missing[:2]} "
              f"structure_ok:{structure}")
        f_ = []
        if num["hard"]:   f_.append("remove numbers absent from findings: " + ", ".join(num["hard"][:5]))
        if bind_bad:      f_.append("wrong number for that metric — use narrative_facts: " + "; ".join(bind_bad[:3]))
        if degen_bad:     f_.append("both sides of the comparison are identical: " + "; ".join(degen_bad[:2]))
        if causal_bad:    f_.append("do not call these a driver/cause: " + ", ".join(causal_bad))
        if scope_bad:     f_.append("remove out-of-scope recommendations: " + ", ".join(scope_bad))
        if raw_feat:      f_.append("use human_label instead of raw field names: " + ", ".join(raw_feat[:3]))
        if not structure: f_.append("start with a '## Executive Summary' heading, plain Markdown, no code fences")
        fixes = "; ".join(f_)
    prompt += f"\n\nREVISE your previous draft with MINIMAL edits. Fix exactly: {fixes}"

if report_md is None and best["text"] is not None and best["score"] < 100:
    report_md = repair_soft_numbers(best["text"], best["num"]["soft"])
    report_source = f"llm best-of-{MAX_ATTEMPTS}+repaired ({LLM_MODEL})"
    print(f"[fallback] no fully clean draft; shipping best attempt (score={best['score']})")

if report_md is None:
    findings["_token_fn"] = page_token
    report_md = inject_segment_urls(
        render_page_tokens(render_fallback_report_v5(findings), PAGE_GROUP_URLS, PAGE_GROUP_LABELS),
        PAGE_GROUP_URLS, PAGE_GROUP_LABELS, focus_segments)

# final assertion: no focus section may ship without its example URL
assert not verify_urls_present(report_md, PAGE_GROUP_URLS, focus_segments), "example URL missing"
print(f"report source: {report_source}\n")
print(report_md)

In [ ]:
# Cell 13 — Step I: email assembly + send (env-configured, DRY_RUN by default)
email_subject = (f"[{severity.upper()}] {timer_name} p75 "
                 f"{win['normal']['p75']} -> {win['anomaly']['p75']}ms "
                 f"({win['delta_p75_pct']:+}%) — {verdict_code.replace('_',' ')}")
email_plain = "\n".join([f"# {timer_name} anomaly report", "",
                          findings["headline"]["transition_sentence"], "",
                          report_md])
email_html = md_to_html(email_plain)
print("subject:", email_subject)

if DRY_RUN_EMAIL:
    Path("report_preview.html").write_text(email_html)
    print("DRY RUN — email not sent; preview written to report_preview.html")
else:
    import boto3
    def _load_conf(path):
        conf = {}
        for raw in Path(path).read_text().splitlines():
            line = raw.strip()
            if line and not line.startswith("#") and "=" in line:
                k, _, v = line.partition("=")
                conf[k.strip()] = v.strip()
        return conf
    sec  = _load_conf("/opt/perf-analytics/.sec/aws-ses")
    mail = _load_conf("/opt/perf-analytics/config/ses_email.conf")
    ses = boto3.client("sesv2", region_name=mail.get("SES_REGION", "ap-northeast-1"),
                       aws_access_key_id=sec["AWS_ACCESS_KEY_ID"],
                       aws_secret_access_key=sec["AWS_SECRET_ACCESS_KEY"])
    resp = ses.send_email(
        FromEmailAddress=mail["SES_FROM_EMAIL"],
        Destination={"ToAddresses": [mail["SES_TO_EMAIL"]]},
        Content={"Simple": {
            "Subject": {"Data": email_subject, "Charset": "UTF-8"},
            "Body": {"Text": {"Data": email_plain, "Charset": "UTF-8"},
                     "Html": {"Data": email_html, "Charset": "UTF-8"}}}})
    print("sent:", resp["MessageId"])

## Upstream requirements (outside this notebook)

1. **Sidecar metadata** — the CSV exporter must also write `<csv_stem>.meta.json`:
```json
{"windows": {"normal":  {"count": 0, "mean": 0, "p50": 0, "p75": 0, "p90": 0, "p95": 0, "p99": 0},
             "anomaly": {"count": 0, "mean": 0, "p50": 0, "p75": 0, "p90": 0, "p95": 0, "p99": 0}},
 "sampling": {"method": "stratified", "strata": "label x timer_decile", "rate": 1.0, "seed": 42},
 "window_boundaries": {"normal": ["...","..."], "anomaly": ["...","..."]}}
```
2. **Stratified sampling** — if sampling is needed, stratify by `label × timer decile` so percentiles survive; the Cell-5 gate blocks the run otherwise.
3. **Timestamp column** — add beacon timestamps to the export to enable window verification and minute-level trend analysis.
4. **Credential hygiene** — the FTP password that appeared in v1 must be rotated; all credentials live in `/opt/perf-analytics/.sec/` with restricted permissions.